# 🛰️ Satellite Crop Type Classification on Azure
## Student Workbook — Wallonia, Belgium 2023

---

This notebook guides you through the full pipeline from Sentinel-2 data to a crop type classification map.  
**The code cells are intentionally incomplete — you need to fill them in.**

Each cell tells you:
- 🎯 **What** to implement
- 💡 **Hints** where needed
- ✅ **Expected output** so you know if it worked

Use the full tutorial document as a reference if you get stuck.

---

### Study Area
| Parameter | Value |
|---|---|
| Region | Wallonia, Belgium |
| Projection | EPSG:32631 — WGS84 / UTM zone 31N |
| Bounding box (UTM) | Easting 610,000–650,030 m \| Northing 5,571,802–5,602,572 m |
| API bbox (WGS84) | [4.55, 50.25, 5.15, 50.55] |
| Imagery | Sentinel-2 L2A, 2023, cloud < 20% |

### Files provided by your trainer
- `config.py` — pre-filled configuration (add your own `STORAGE_KEY`)
- `wallonia_insitu_2023.shp` + companions — training polygons (LPIS + Walous 2023)
- `crop_dictionary.csv` — class code lookup table
- `test_auth.py`, `test_search.py`, `download_s2.py` — ready-to-run scripts

---

---
# DAY 1 — Data Access & Verification
---

## Step 1 — Load Configuration

Upload `config.py` to JupyterLab (drag and drop into the file browser).  
Open it and fill in your `STORAGE_KEY` (from your Azure storage account → Access keys → key1).

🎯 Import the config and verify all variables are loaded.

In [3]:
# Import all variables from config.py
# YOUR CODE HERE
from config import *
# Verify config loaded correctly — print the storage account name, ROI bbox, and number of bands
# YOUR CODE HERE
print(f"Storage account: {STORAGE_ACCOUNT}")
print(f"ROI: {ROI_BBOX}")
print(f"Bands: {len(BANDS)}")
# ✅ Expected output:
# Storage account: s2wallonia2023
# ROI: [4.55, 50.25, 5.15, 50.55]
# Bands: 11

Storage account: mycropdatakh
ROI: [4.55, 50.25, 5.15, 50.55]
Bands: 11


In [ ]:
import sys
!{sys.executable} -m pip install rasterio azure-storage-blob pystac-client geopandas scikit-learn scipy


## Step 2 — Test Copernicus Authentication

Run `test_auth.py` in the Terminal first:  
```bash
python3 test_auth.py
```

Then implement the authentication function below.  

🎯 Write a function `get_token()` that authenticates with the Copernicus identity server and returns an access token.

💡 The token endpoint is: `https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token`  
💡 You need to POST with `client_id='cdse-public'`, `grant_type='password'`, `username`, `password`  
💡 The token is in `response.json()['access_token']`

In [10]:
import requests

def get_token():
    """
    Authenticate with Copernicus Data Space and return an access token.
    The token expires after 10 minutes.
    """
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    response = requests.post(url, data={
        "client_id":   "cdse-public",
        "grant_type":  "password",
        "username":    COPERNICUS_USER,
        "password":    COPERNICUS_PASS,
    })
    response.raise_for_status()
    return response.json()["access_token"]

# Test it
token = get_token()
print(f'Token obtained: {token[:50]}...')

# ✅ Expected: Token obtained: eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVC...

Token obtained: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6IC...


## Step 3 — Search the STAC Catalogue

Run `test_search.py` in the Terminal to verify the search works.  
Then implement it yourself below.

🎯 Search the Copernicus STAC catalogue for Sentinel-2 L2A scenes over your ROI.

💡 STAC endpoint: `https://catalogue.dataspace.copernicus.eu/stac`  
💡 Collection name: `sentinel-2-l2a` (not 'SENTINEL-2')  
💡 Use `pystac_client.Client.open()` then `.search()` with `collections`, `bbox`, `datetime`, `query`  
💡 Filter cloud cover with `query={'eo:cloud_cover': {'lt': MAX_CLOUD}}`

In [ ]:
import sys
!{sys.executable} -m pip install pystac-client

In [12]:
from pystac_client import Client
from collections import Counter

# Connect to the STAC catalogue and search for scenes
# YOUR CODE HERE
catalog = Client.open("https://catalogue.dataspace.copernicus.eu/stac")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=ROI_BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}}
)
# Store results in a variable called 'items'
# YOUR CODE HERE
items = list(search.items())
# Print: total number of scenes found
# Print: number of scenes per month
# Print: list of available asset keys in the first scene
# YOUR CODE HERE
print(f"Found {len(items)} scenes")
months = Counter(item.datetime.strftime("%Y-%m") for item in items)
for month, count in sorted(months.items()):
    print(f"  {month}: {count} scenes")
print("\nAvailable assets:")
for key in items[0].assets:
    print(f"  {key}: ✅")
# ✅ Expected:
# Found 39 scenes
# 2023-02: 3 scenes  ...  2023-10: 2 scenes
# B02_10m: ✅  B03_10m: ✅  ...  SCL_20m: ✅

Found 39 scenes
  2023-02: 3 scenes
  2023-03: 2 scenes
  2023-04: 2 scenes
  2023-05: 2 scenes
  2023-06: 16 scenes
  2023-07: 1 scenes
  2023-08: 5 scenes
  2023-09: 6 scenes
  2023-10: 2 scenes

Available assets:
  AOT_10m: ✅
  AOT_20m: ✅
  AOT_60m: ✅
  B01_20m: ✅
  B01_60m: ✅
  B02_10m: ✅
  B02_20m: ✅
  B02_60m: ✅
  B03_10m: ✅
  B03_20m: ✅
  B03_60m: ✅
  B04_10m: ✅
  B04_20m: ✅
  B04_60m: ✅
  B05_20m: ✅
  B05_60m: ✅
  B06_20m: ✅
  B06_60m: ✅
  B07_20m: ✅
  B07_60m: ✅
  B08_10m: ✅
  B09_60m: ✅
  B11_20m: ✅
  B11_60m: ✅
  B12_20m: ✅
  B12_60m: ✅
  B8A_20m: ✅
  B8A_60m: ✅
  CLD_20m: ✅
  CLD_60m: ✅
  Product: ✅
  SCL_20m: ✅
  SCL_60m: ✅
  SNW_20m: ✅
  SNW_60m: ✅
  TCI_10m: ✅
  TCI_20m: ✅
  TCI_60m: ✅
  WVP_10m: ✅
  WVP_20m: ✅
  WVP_60m: ✅
  thumbnail: ✅
  safe_manifest: ✅
  granule_metadata: ✅
  inspire_metadata: ✅
  product_metadata: ✅
  datastrip_metadata: ✅


## Step 4 — Download Sentinel-2 Data

Run `download_s2.py` in a Terminal — do not run it in a notebook cell as it takes 2–4 hours:  
```bash
python3 download_s2.py
```

While it runs, implement the Azure Blob Storage helper functions below — you will need them from Day 2 onwards.

🎯 Write three functions: `get_cc()`, `read_blob()`, `write_blob()`

💡 `BlobServiceClient.from_connection_string(conn_str)` creates the client  
💡 Use `MemoryFile` from rasterio to read/write rasters in memory without saving to disk  
💡 Connection string format: `DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net`

In [4]:
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from azure.storage.blob import BlobServiceClient

NO_DATA = -9999.0

def get_cc():
    """
    Return an Azure Blob container client using credentials from config.
    """
    # Build connection string and return container client
    # YOUR CODE HERE
    conn_str = (
        f"DefaultEndpointsProtocol=https;"
        f"AccountName={STORAGE_ACCOUNT};"
        f"AccountKey={STORAGE_KEY};"
        f"EndpointSuffix=core.windows.net"
    )
    client = BlobServiceClient.from_connection_string(conn_str)
    return client.get_container_client(CONTAINER_NAME)


def read_blob(blob_name):
    """
    Download a raster file from Blob Storage and return it as a numpy array.
    Returns: (array, rasterio_profile)
    """
    # 1. Download the blob bytes using get_cc()
    # 2. Open with MemoryFile and rasterio
    # 3. Read band 1 as float32
    # 4. Return (array, profile)
    # YOUR CODE HERE
    data = get_cc().download_blob(blob_name).readall()
    with MemoryFile(data) as memfile:
        with memfile.open() as ds:
            arr = ds.read(1).astype("float32")
            profile = ds.profile
    return arr, profile


def write_blob(arr, profile, blob_name):
    prof = profile.copy()
    prof.update(dtype="float32", count=1, nodata=NO_DATA, driver="GTiff")
    with MemoryFile() as memfile:
        with memfile.open(**prof) as ds:
            ds.write(arr.astype("float32"), 1)
        get_cc().upload_blob(blob_name, memfile.read(), overwrite=True)
    print(f"Saved: {blob_name}")



# Test: list files in Blob Storage
cc    = get_cc()
blobs = [b.name for b in cc.list_blobs(name_starts_with='raw/')]
print(f'Files in Blob Storage: {len(blobs)}')

# Count files per band
import re
from collections import Counter
band_counts = Counter()
for b in blobs:
    m = re.search(r'_(B\w+|SCL_\w+)\.tif$', b)
    if m: band_counts[m.group(1)] += 1
print('Files per band:')
for band, n in sorted(band_counts.items()):
    print(f'  {band:10s}: {n} scenes')

# ✅ Expected: ~39 files per band (may be less if download still running)

Files in Blob Storage: 242
Files per band:
  B02_10m   : 22 scenes
  B03_10m   : 22 scenes
  B04_10m   : 22 scenes
  B05_20m   : 22 scenes
  B06_20m   : 22 scenes
  B07_20m   : 22 scenes
  B08_10m   : 22 scenes
  B11_20m   : 22 scenes
  B12_20m   : 22 scenes
  B8A_20m   : 22 scenes
  SCL_20m   : 22 scenes


---
# DAY 2 — Cloud Masking, Compositing & Feature Stack
---

## Step 5 — Group Scenes by Month

🎯 Build a nested dictionary `monthly` that groups blob paths by month and band.  
Structure: `monthly['202306']['B04_10m'] = ['raw/T31UFS/20230601_B04_10m.tif', ...]`

💡 Use a regex on the blob name to extract the 6-digit month (YYYYMM) and band name  
💡 `defaultdict(lambda: defaultdict(list))` is useful here

In [4]:
from collections import defaultdict
import re

# First get the list of all blobs
blobs = [b.name for b in get_cc().list_blobs(name_starts_with='raw/')]

# Build the monthly grouping dictionary
monthly = defaultdict(lambda: defaultdict(list))

for blob in blobs:
    # Extract YYYYMM and band name from filename like: raw/T31UFS/20230615_B04_10m.tif
    m = re.search(r'/(\d{6})\d{2}_(B\w+|SCL_\w+)\.tif$', blob)
    if m:
        month = m.group(1)   # e.g. '202306'
        band  = m.group(2)   # e.g. 'B04_10m'
        monthly[month][band].append(blob)

# Print months available and B04 scene counts
print("Months:", sorted(monthly.keys()))
for month in sorted(monthly.keys()):
    n = len(monthly[month].get('B04_10m', []))
    print(f"  {month}: {n} B04 scenes")


# Build the monthly grouping dictionary
# YOUR CODE HERE

# Print: months available and number of B04_10m scenes per month
# YOUR CODE HERE

# ✅ Expected:
# Months: ['202302', '202303', ..., '202310']
# 202302: 3 B04 scenes   202306: 16 B04 scenes   etc.

Months: ['202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310']
  202302: 2 B04 scenes
  202303: 1 B04 scenes
  202304: 1 B04 scenes
  202305: 2 B04 scenes
  202306: 8 B04 scenes
  202307: 1 B04 scenes
  202308: 3 B04 scenes
  202309: 3 B04 scenes
  202310: 1 B04 scenes


## Step 6 — Cloud Masking Function

The SCL (Scene Classification Layer) labels each pixel. We keep only valid pixels.

| SCL value | Meaning | Keep? |
|---|---|---|
| 4 | Vegetation | ✅ |
| 5 | Non-vegetated | ✅ |
| 6 | Water | ✅ |
| 7 | Unclassified | ✅ |
| 11 | Snow/Ice | ✅ |
| All others | Cloud, shadow, saturated | ❌ |

🎯 Apply the SCL mask inside the compositing loop: set masked pixels to `np.nan`.

💡 `np.isin(scl_arr.astype(int), VALID_SCL)` creates a boolean mask  
💡 `np.where(mask, arr, np.nan)` applies it

## Step 7 — Monthly Median Composites

🎯 For each month and each feature band:  
1. Load the SCL array for that month  
2. Load all scenes for that band  
3. Apply cloud mask to each scene  
4. Stack scenes and compute pixel-wise median (NaN-aware)  
5. Replace remaining NaN with NO_DATA  
6. Save composite to `composites/{month}_{band}.tif` in Blob Storage

💡 `np.nanmedian(np.stack(scene_list, axis=0), axis=0)` computes the NaN-aware median  
💡 Feature bands = all BANDS except `SCL_20m`  
💡 Store the first valid `profile` as `ref_profile` — you'll need it later

In [27]:
VALID_SCL  = [4, 5, 6, 7, 11]
FEAT_BANDS = [b for b in BANDS if b != 'SCL_20m']
ref_profile = None

for month in sorted(monthly.keys()):
    print(f'\nProcessing {month}...')

    # 1. Load SCL for this month
    scl_blobs = monthly[month].get('SCL_20m', [])
    if not scl_blobs:
        print(f'  No SCL for {month}, skipping')
        continue
    scl_arr, _ = read_blob(scl_blobs[0])

    for band in FEAT_BANDS:
        band_blobs = monthly[month].get(band, [])
        if not band_blobs:
            print(f'  {band}: no data, skipping')
            continue

        stack = []
        for blob in band_blobs:
            # 2. Load band array
            arr, profile = read_blob(blob)

            # Store first valid profile
            if ref_profile is None:
                ref_profile = profile

            # Upsample SCL to match band resolution if needed
            if scl_arr.shape != arr.shape:
                factor = arr.shape[0] // scl_arr.shape[0]
                scl_up = np.repeat(np.repeat(scl_arr, factor, axis=0), factor, axis=1)
            else:
                scl_up = scl_arr

            # 3. Apply cloud mask
            mask   = np.isin(scl_up.astype(int), VALID_SCL)
            masked = np.where(mask, arr, np.nan)

            # 4. Append to stack
            stack.append(masked)

        # 5. Compute NaN-aware median across all scenes
        composite = np.nanmedian(np.stack(stack, axis=0), axis=0)

        # 6. Replace remaining NaN with NO_DATA and save
        composite = np.where(np.isnan(composite), NO_DATA, composite)
        out_name  = f'composites/{month}_{band}.tif'
        write_blob(composite, profile, out_name)

print('\nAll composites done!')




Processing 202302...
Saved: composites/202302_B02_10m.tif
Saved: composites/202302_B03_10m.tif
Saved: composites/202302_B04_10m.tif
Saved: composites/202302_B05_20m.tif
Saved: composites/202302_B06_20m.tif
Saved: composites/202302_B07_20m.tif
Saved: composites/202302_B08_10m.tif
Saved: composites/202302_B8A_20m.tif
Saved: composites/202302_B11_20m.tif
Saved: composites/202302_B12_20m.tif

Processing 202303...
Saved: composites/202303_B02_10m.tif
Saved: composites/202303_B03_10m.tif
Saved: composites/202303_B04_10m.tif
Saved: composites/202303_B05_20m.tif
Saved: composites/202303_B06_20m.tif
Saved: composites/202303_B07_20m.tif
Saved: composites/202303_B08_10m.tif
Saved: composites/202303_B8A_20m.tif
Saved: composites/202303_B11_20m.tif
Saved: composites/202303_B12_20m.tif

Processing 202304...
Saved: composites/202304_B02_10m.tif
Saved: composites/202304_B03_10m.tif
Saved: composites/202304_B04_10m.tif
Saved: composites/202304_B05_20m.tif
Saved: composites/202304_B06_20m.tif
Saved: co

## Step 8 — Spectral Indices

Compute three indices for each month to enrich your feature set.

| Index | Formula | What it captures |
|---|---|---|
| NDVI | (B08 - B04) / (B08 + B04) | Vegetation density and health |
| NDWI | (B03 - B08) / (B03 + B08) | Water bodies, irrigation |
| NDBI | (B11 - B08) / (B11 + B08) | Built-up surfaces |

🎯 For each month: load the relevant composites, compute the three indices, save each to Blob Storage.

💡 Avoid division by zero: use `np.where(denominator != 0, (a-b)/(a+b), NO_DATA)`  
💡 Clip values to [-1, 1] — NDVI outside this range indicates a bad pixel

In [6]:
def safe_index(a, b):
    """
    Compute a normalised difference index: (a - b) / (a + b)
    Returns NO_DATA where denominator is zero.
    Clips result to [-1, 1].
    """
    denom = a + b
    result = np.where(denom != 0, (a - b) / denom, NO_DATA)
    return np.clip(result, -1, 1)


for month in sorted(monthly.keys()):
    print(f'Indices for {month}...')

    # Load B03, B04, B08, B11 composites for this month
    # Replace NO_DATA with np.nan before computing indices
    b03, profile = read_blob(f'composites/{month}_B03_10m.tif')
    b04, _       = read_blob(f'composites/{month}_B04_10m.tif')
    b08, _       = read_blob(f'composites/{month}_B08_10m.tif')
    b11, _       = read_blob(f'composites/{month}_B11_20m.tif')

    for arr in [b03, b04, b08, b11]:
        arr[arr == NO_DATA] = np.nan

    if b11.shape != b08.shape:
        factor = b08.shape[0] // b11.shape[0]
        b11 = np.repeat(np.repeat(b11, factor, axis=0), factor, axis=1)

    # Compute and save NDVI, NDWI, NDBI
    ndvi = safe_index(b08, b04)
    ndwi = safe_index(b03, b08)
    ndbi = safe_index(b11, b08)

    write_blob(ndvi, profile, f'composites/{month}_NDVI.tif')
    write_blob(ndwi, profile, f'composites/{month}_NDWI.tif')
    write_blob(ndbi, profile, f'composites/{month}_NDBI.tif')

print('Spectral indices done!')

# ✅ Expected: 3 index files per month saved to composites/
# e.g. composites/202306_NDVI.tif  composites/202306_NDWI.tif  composites/202306_NDBI.tif

Indices for 202302...
Saved: composites/202302_NDVI.tif
Saved: composites/202302_NDWI.tif
Saved: composites/202302_NDBI.tif
Indices for 202303...
Saved: composites/202303_NDVI.tif
Saved: composites/202303_NDWI.tif
Saved: composites/202303_NDBI.tif
Indices for 202304...
Saved: composites/202304_NDVI.tif
Saved: composites/202304_NDWI.tif
Saved: composites/202304_NDBI.tif
Indices for 202305...
Saved: composites/202305_NDVI.tif
Saved: composites/202305_NDWI.tif
Saved: composites/202305_NDBI.tif
Indices for 202306...
Saved: composites/202306_NDVI.tif
Saved: composites/202306_NDWI.tif
Saved: composites/202306_NDBI.tif
Indices for 202307...
Saved: composites/202307_NDVI.tif
Saved: composites/202307_NDWI.tif
Saved: composites/202307_NDBI.tif
Indices for 202308...
Saved: composites/202308_NDVI.tif
Saved: composites/202308_NDWI.tif
Saved: composites/202308_NDBI.tif
Indices for 202309...


/tmp/ipykernel_6070/2974399620.py:8: RuntimeWarning: invalid value encountered in divide
  result = np.where(denom != 0, (a - b) / denom, NO_DATA)


Saved: composites/202309_NDVI.tif
Saved: composites/202309_NDWI.tif
Saved: composites/202309_NDBI.tif
Indices for 202310...
Saved: composites/202310_NDVI.tif
Saved: composites/202310_NDWI.tif
Saved: composites/202310_NDBI.tif
Spectral indices done!


## Step 9 — Assemble the Feature Stack

🎯 Load all composite files from Blob Storage and stack them into a single 3D NumPy array.

The result should have shape `(rows, cols, n_features)` where `n_features` is the total number of band-month combinations.

💡 `np.dstack(list_of_2d_arrays)` stacks along the third axis  
💡 Save `feat_names` (the ordered list of feature names) to a JSON file — you'll need it on Day 3

In [7]:
import json
import numpy as np
from rasterio.io import MemoryFile
from rasterio.windows import from_bounds
from pyproj import Transformer

transformer = Transformer.from_crs(4326, 32631, always_xy=True)
left,  bottom = transformer.transform(ROI_BBOX[0], ROI_BBOX[1])
right, top    = transformer.transform(ROI_BBOX[2], ROI_BBOX[3])
roi_bounds = (left, bottom, right, top)

def read_blob_roi(blob_name):
    data = get_cc().download_blob(blob_name).readall()
    with MemoryFile(data) as memfile:
        with memfile.open() as ds:
            window  = from_bounds(*roi_bounds, ds.transform)
            arr     = ds.read(1, window=window).astype("float32")
            profile = ds.profile.copy()
            profile.update(width=arr.shape[1], height=arr.shape[0],
                           transform=ds.window_transform(window))
    return arr, profile

comp_blobs = sorted([b.name for b in get_cc().list_blobs(name_starts_with='composites/')])

arr0, ref_profile = read_blob_roi(comp_blobs[0])
rows, cols = arr0.shape
n_feat = len(comp_blobs)

feat_arr   = np.empty((rows, cols, n_feat), dtype=np.float32)
feat_names = []

feat_arr[:, :, 0] = arr0
feat_names.append(comp_blobs[0].replace('composites/', '').replace('.tif', ''))

for i, blob in enumerate(comp_blobs[1:], start=1):
    arr, _ = read_blob_roi(blob)
    if arr.shape != (rows, cols):
        factor = rows // arr.shape[0]
        arr = np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1)
        out = np.full((rows, cols), NO_DATA, dtype=np.float32)
        r = min(arr.shape[0], rows)
        c = min(arr.shape[1], cols)
        out[:r, :c] = arr[:r, :c]
        arr = out
    feat_arr[:, :, i] = arr
    feat_names.append(blob.replace('composites/', '').replace('.tif', ''))
    if i % 10 == 0:
        print(f'  Loaded {i}/{n_feat}...')

print(f'Feature stack shape: {feat_arr.shape}')
print(f'Number of features: {len(feat_names)}')
print(f'First 5 features: {feat_names[:5]}')

with open('feature_names.json', 'w') as f:
    json.dump(feat_names, f)
print('Saved feature_names.json')



# ✅ Expected:
# Feature stack shape: (3100, 4000, 156)  <- rows × cols × features
# First features: ['202302_B02_10m', '202302_B03_10m', ...]

  Loaded 10/117...
  Loaded 20/117...
  Loaded 30/117...
  Loaded 40/117...
  Loaded 50/117...
  Loaded 60/117...
  Loaded 70/117...
  Loaded 80/117...
  Loaded 90/117...
  Loaded 100/117...
  Loaded 110/117...
Feature stack shape: (3246, 4181, 117)
Number of features: 117
First 5 features: ['202302_B02_10m', '202302_B03_10m', '202302_B04_10m', '202302_B05_20m', '202302_B06_20m']
Saved feature_names.json


---
# DAY 3 — Feature Selection & Classification
---

## Step 10 — Reload Feature Stack

Start a fresh notebook or restart the kernel. Reload everything from Blob Storage.

🎯 Reload `feat_arr` and `feat_names` from Blob Storage and `feature_names.json`.

In [ ]:
# Paste your helper functions (get_cc, read_blob, write_blob) here
# YOUR CODE HERE

# Load feature names from feature_names.json
# YOUR CODE HERE

# Reload all composites and rebuild feat_arr
# (same code as Step 9 — you can copy it)
# YOUR CODE HERE

# Reload config and imports
from config import *
import numpy as np
import json
import rasterio
from rasterio.io import MemoryFile
from rasterio.windows import from_bounds
from azure.storage.blob import BlobServiceClient
from pyproj import Transformer

NO_DATA = -9999.0

# Paste your helper functions
def get_cc():
    conn_str = (
        f"DefaultEndpointsProtocol=https;"
        f"AccountName={STORAGE_ACCOUNT};"
        f"AccountKey={STORAGE_KEY};"
        f"EndpointSuffix=core.windows.net"
    )
    client = BlobServiceClient.from_connection_string(conn_str)
    return client.get_container_client(CONTAINER_NAME)

def read_blob(blob_name):
    data = get_cc().download_blob(blob_name).readall()
    with MemoryFile(data) as memfile:
        with memfile.open() as ds:
            arr = ds.read(1).astype("float32")
            profile = ds.profile
    return arr, profile

def write_blob(arr, profile, blob_name):
    prof = profile.copy()
    prof.update(dtype="float32", count=1, nodata=NO_DATA, driver="GTiff")
    with MemoryFile() as memfile:
        with memfile.open(**prof) as ds:
            ds.write(arr.astype("float32"), 1)
        get_cc().upload_blob(blob_name, memfile.read(), overwrite=True)
    print(f"Saved: {blob_name}")

transformer = Transformer.from_crs(4326, 32631, always_xy=True)
left, bottom = transformer.transform(ROI_BBOX[0], ROI_BBOX[1])
right, top = transformer.transform(ROI_BBOX[2], ROI_BBOX[3])
roi_bounds = (left, bottom, right, top)

def read_blob_roi(blob_name):
    data = get_cc().download_blob(blob_name).readall()
    with MemoryFile(data) as memfile:
        with memfile.open() as ds:
            window = from_bounds(*roi_bounds, ds.transform)
            arr = ds.read(1, window=window).astype("float32")
            profile = ds.profile.copy()
            profile.update(width=arr.shape[1], height=arr.shape[0],
                           transform=ds.window_transform(window))
    return arr, profile

# Load feature names
with open('feature_names.json', 'r') as f:
    feat_names = json.load(f)

# Rebuild feature stack from composites
comp_blobs = sorted([b.name for b in get_cc().list_blobs(name_starts_with='composites/')])

arr0, ref_profile = read_blob_roi(comp_blobs[0])
rows, cols = arr0.shape
n_feat = len(comp_blobs)

feat_arr = np.empty((rows, cols, n_feat), dtype=np.float32)
feat_arr[:, :, 0] = arr0

for i, blob in enumerate(comp_blobs[1:], start=1):
    arr, _ = read_blob_roi(blob)
    if arr.shape != (rows, cols):
        factor = rows // arr.shape[0]
        arr = np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1)
        out = np.full((rows, cols), NO_DATA, dtype=np.float32)
        r = min(arr.shape[0], rows)
        c = min(arr.shape[1], cols)
        out[:r, :c] = arr[:r, :c]
        arr = out
    feat_arr[:, :, i] = arr
    if i % 10 == 0:
        print(f'  Loaded {i}/{n_feat}...')

print(f'Feature stack: {feat_arr.shape}')

Feature stack: (3246, 4181, 117)


  Loaded 10/117...
  Loaded 20/117...


## Step 11 — Rasterize Training Data

Upload the training shapefile to JupyterLab (drag and drop all 4 files: `.shp`, `.dbf`, `.shx`, `.prj`).

🎯 Load the shapefile, check/fix the CRS, and rasterize the training polygons onto the same grid as your feature stack.

💡 Use `gpd.read_file()` to load, `.to_crs()` to reproject if needed  
💡 Use `rasterio.features.rasterize()` with `out_shape`, `fill`, `transform` from `ref_profile`  
💡 The field with crop/landcover codes is `grp_1_nb`

In [ ]:
import geopandas as gpd
from rasterio.features import rasterize as rio_rasterize

IN_SITU_SHP = 'wallonia_insitu_2023.shp'
FIELD_CODE  = 'grp_1_nb'

# 1. Load the shapefile
# YOUR CODE HERE

# 2. Print: number of polygons, CRS, unique crop codes
# YOUR CODE HERE

# 3. Check CRS matches raster — reproject if not
# YOUR CODE HERE

# 4. Rasterize: burn FIELD_CODE values onto a grid matching feat_arr
# Use fill=int(NO_DATA) for pixels outside polygons
# YOUR CODE HERE

# 5. Print: number of training pixels and number of classes
# YOUR CODE HERE

# ✅ Expected:
# Training pixels: ~35,000+
# Classes: 15–25 unique crop/landcover types

## Step 12 — Build X and y, Train/Test Split

🎯 Extract the feature matrix X and label vector y from pixels that have training labels.

💡 Boolean mask: `mask = cal_arr != int(NO_DATA)`  
💡 `X = feat_arr[mask, :]` — shape (n_samples, n_features)  
💡 Use `train_test_split` with `stratify=y` to ensure all classes appear in both sets

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Create boolean mask of labeled pixels
# YOUR CODE HERE

# 2. Extract X (features) and y (labels)
# YOUR CODE HERE

# 3. Replace NaN in X with NO_DATA
# YOUR CODE HERE

# 4. Stratified 80/20 train/test split
# YOUR CODE HERE

# 5. Print shapes
# YOUR CODE HERE

# ✅ Expected:
# X shape: (n_samples, n_features)
# Training: ~28,000  |  Test: ~7,000

## Step 13 — Feature Selection

With up to 156 features, not all are equally useful. Use a quick Random Forest to rank them.

🎯 Train a fast Random Forest on a 10,000-pixel subsample and rank features by Gini importance.

**Group discussion:** After plotting the importances, answer these questions before selecting your subset:
- Which months are most important? Does this make agronomic sense for Wallonia?
- Which bands appear most often in the top features?
- Are the spectral indices (NDVI, NDWI, NDBI) useful?
- How many features do you need before importance drops off significantly?

💡 Use `n_estimators=50` for speed  
💡 `rf.feature_importances_` gives the Gini importance of each feature

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# 1. Subsample 10,000 training pixels randomly
# YOUR CODE HERE

# 2. Train a quick Random Forest (50 trees) on the subsample
# YOUR CODE HERE

# 3. Build a DataFrame of feature importances, sorted descending
# YOUR CODE HERE

# 4. Print top 15 features
# YOUR CODE HERE

# 5. Plot top 30 as a bar chart, save as feature_importance.png
# YOUR CODE HERE

# ✅ Expected: bar chart showing top 30 features by importance

In [ ]:
# FEATURE SELECTION — discuss with your group before running this cell

# How many features will you keep? Justify your choice:
# (Consider: accuracy vs training speed, which features matter most)
TOP_N = None  # <-- set this after your group discussion

# Select top N features
# YOUR CODE HERE

# Create subsetted versions of your training/test matrices and the image
# X_tr_sel, X_te_sel, img_flat_sel
# YOUR CODE HERE

print(f'Selected {TOP_N} features for training.')
print(f'Top features: {selected[:10]}')

## Step 14 — Train the Random Forest

🎯 Train a full Random Forest classifier on the selected features.

💡 Use `n_estimators=100`, `oob_score=True`, `n_jobs=-1` (uses all CPU cores)  
💡 OOB (Out-of-Bag) score is a free accuracy estimate on data not seen by each tree — a useful sanity check  
💡 Aim for OOB > 80%. If lower, revisit your feature selection or check training data quality.

In [ ]:
import time

# Train the Random Forest
# Print training time and OOB accuracy
# YOUR CODE HERE

# ✅ Expected:
# Training time: 30–120s depending on n_features and n_samples
# OOB accuracy: ideally > 80%

## Step 15 — Predict the Full Image

🎯 Apply the trained classifier to every pixel in the image.

💡 The image needs to be reshaped from `(rows, cols, n_features)` to `(rows*cols, n_features)` for prediction  
💡 After prediction, reshape back to `(rows, cols)` to get a 2D classification map  
💡 Replace NaN values with NO_DATA before predicting

In [ ]:
# 1. Reshape image and replace NaN
# YOUR CODE HERE

# 2. Predict (time this — it should take 20–60s on DS3_v2)
# YOUR CODE HERE

# 3. Reshape prediction back to (rows, cols)
# YOUR CODE HERE

# 4. Save to Blob Storage at results/classification_RF.tif
# YOUR CODE HERE

# ✅ Expected:
# Prediction complete in ~30s
# pred_map shape: (rows, cols)
# Unique predicted classes: [3, 21, 69, ...]

---
# DAY 4 — Evaluation & Post-Processing
---

## Step 16 — Accuracy Assessment

🎯 Evaluate your classifier on the held-out test set (20% of training pixels not seen during training).

Report:
- Overall Accuracy (OA)
- Per-class Precision, Recall, F1-score

**Reflection questions for your group:**
- Which classes have the lowest F1-score? Why might this be?
- How does OA compare to the OOB score from training?
- If a class has high precision but low recall, what does that mean?

💡 `classification_report(y_te, y_pred)` gives all per-class metrics in one call

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Predict on test set and compute accuracy
# YOUR CODE HERE

# ✅ Expected: Overall Accuracy + per-class report table

## Step 17 — Reclassify with Crop Dictionary

The classification map uses detailed numeric codes. The crop dictionary maps these to broader, more readable crop groups.

🎯 Load `crop_dictionary.csv` and remap the detailed codes to broader group codes.

💡 Columns: `grp_1_nb` (detailed code) → `grp_A_nb` (broader code) → `grp_A` (name)  
💡 Loop through the LUT rows and replace values in a copy of `pred_map`

In [ ]:
# 1. Load crop_dictionary.csv (upload to JupyterLab first)
# YOUR CODE HERE

# 2. Load classification map from Blob Storage
# YOUR CODE HERE

# 3. Remap detailed codes to broad group codes using the LUT
# YOUR CODE HERE

# 4. Save reclassified map to Blob Storage
# YOUR CODE HERE

# ✅ Expected: reclassification_RF.tif saved to results/ in Blob Storage

## Step 18 — Majority Filter

Classification maps often have 'salt and pepper' noise — isolated pixels classified differently from their neighbours. A majority filter replaces each pixel with the most common class in its neighbourhood.

🎯 Apply a 3×3 majority (mode) filter to smooth the reclassified map.

💡 Use `scipy.ndimage.generic_filter()` with a custom mode function  
💡 `scipy.stats.mode(x.astype(int), keepdims=False).mode` returns the most common value  
💡 Use `mode='nearest'` to handle edges

In [ ]:
from scipy.ndimage import generic_filter
from scipy.stats import mode as sp_mode

def majority_filter(arr, ws=3):
    """
    Apply a majority (mode) filter to a 2D integer array.
    Each pixel is replaced by the most common value in its ws x ws neighbourhood.
    """
    # YOUR CODE HERE
    pass


# Apply filter and save result
print('Applying majority filter...')
# YOUR CODE HERE

# ✅ Expected: reclassification_RF_filtered.tif saved to Blob Storage

## Step 19 — Visualise the Final Map

🎯 Create a publication-quality visualisation of your final classified map with a legend showing crop/land cover names.

💡 Use `matplotlib.colors.ListedColormap` with `plt.cm.tab20` to create a categorical colour map  
💡 Use `matplotlib.patches.Patch` to create legend entries  
💡 Map class codes to names using `code_to_name = dict(zip(lut_df[FIELD_NEW], lut_df[FIELD_NM]))`  
💡 Replace NO_DATA pixels with `np.nan` before plotting so they appear transparent

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# 1. Get unique class codes from filtered_map
# YOUR CODE HERE

# 2. Build code → name mapping from LUT
# YOUR CODE HERE

# 3. Create colour map
# YOUR CODE HERE

# 4. Plot the map with legend
# YOUR CODE HERE

# 5. Save as final_crop_map.png at dpi=200
# YOUR CODE HERE

# ✅ Expected: colourful crop map with legend showing crop/landcover names

## Step 20 — Download Results

🎯 Download your final GeoTIFF results to the JupyterLab local filesystem so you can save them to your laptop.

Then right-click each file in the JupyterLab file browser and select **Download** to save to your local machine.

In [ ]:
def download_blob(blob_name, local_path):
    """
    Download a file from Blob Storage to the local filesystem.
    """
    # YOUR CODE HERE
    pass


download_blob('results/reclassification_RF_filtered.tif', 'final_map.tif')
download_blob('results/classification_RF.tif',            'raw_classification.tif')
print('Results downloaded. Right-click files in the JupyterLab browser to save locally.')

## Step 21 — Clean Up Azure Resources

⚠️ **Do this only after downloading all results you want to keep.**

1. Go to **portal.azure.com**
2. **Resource groups** → your group resource group
3. Click **Delete resource group**
4. Type the resource group name to confirm → **Delete**

This removes all resources (compute instance, storage, ML workspace) and stops all costs permanently.

Your Azure for Students account and remaining $100 credit are **not affected**.

---

## 🎉 Congratulations!

You have built a complete end-to-end satellite image classification pipeline on Azure:

- ✅ Provisioned cloud infrastructure from scratch
- ✅ Downloaded Sentinel-2 multi-band data via the Copernicus S3 API
- ✅ Applied cloud masking and computed monthly composites
- ✅ Built a multi-temporal feature stack with spectral indices
- ✅ Selected informative features using Random Forest importance
- ✅ Trained and evaluated a crop type classifier
- ✅ Produced a final classified land cover map of Wallonia

---